In [1]:
import torch
import tifffile
import numpy as np
import os
from typing import Optional, Tuple, Union
from tqdm import tqdm
from skimage import exposure
from micro_sam.automatic_segmentation import get_predictor_and_segmenter, automatic_instance_segmentation
from scipy.ndimage import label as ndimage_label, generate_binary_structure
from skimage.morphology import remove_small_objects


# Define the custom transform class that was used during training.
# This is necessary for torch.load (unpickling) to find the class definition.
class ChannelSafeIntensityAug:
    def __call__(self, x):
        # This transform operates on numpy arrays
        for c in range(2):
            if np.random.rand() > 0.5:
                # Add gaussian noise
                x[c] = x[c] + np.random.randn(*x[c].shape) * 0.05
            if np.random.rand() > 0.5:
                # Adjust brightness
                factor = np.random.uniform(0.8, 1.2)
                x[c] = np.clip(x[c] * factor, 0, 1)
        return x


def run_automatic_instance_segmentation(
    image: np.ndarray,
    checkpoint_path: Union[os.PathLike, str],
    model_type: str = "vit_b_lm",
    device: Optional[Union[str, torch.device]] = None,
    tile_shape: Optional[Tuple[int, int]] = None,
    halo: Optional[Tuple[int, int]] = None,
    **kwargs,
):
    """Automatic Instance Segmentation (AIS) wrapper for convenience.
    
    Args:
        image: The input image (numpy array).
        checkpoint_path: The path to stored checkpoints.
        model_type: The choice of the µsam model.
        device: The device to run the model inference.
        tile_shape: The tile shape for tiling-based segmentation.
        halo: The overlap shape on each side per tile for stitching tiles.
    
    Returns:
        The instance segmentation result.
    """
    # Get the predictor and segmenter
    predictor, segmenter = get_predictor_and_segmenter(
        model_type=model_type,
        checkpoint=checkpoint_path,
        device=device,
        is_tiled=(tile_shape is not None),
        **kwargs,
    )
    
    # Run automatic instance segmentation
    prediction = automatic_instance_segmentation(
        predictor=predictor,
        segmenter=segmenter,
        input_path=image,
        ndim=2,
        tile_shape=tile_shape,
        halo=halo,
    )
    
    return prediction

In [10]:
import re

# ============================
# 1. CONFIGURATION
# ============================
MODEL_TYPE  = "vit_l_lm"
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"

# ── Change these two lines only ──────────────────────────────────────────────
CHECKPOINT_PATH = r"/work/scratch/rawat/models/microsam_training_output_large/checkpoints/vit_l_lm_neo_augmented_PSF_v2/best.pt"
INPUT_FOLDER    = r"/home/students/rawat/Final_dataset/PSF/test/images/"
# ─────────────────────────────────────────────────────────────────────────────

BLOB_REMOVAL_THRESHOLD = 862
VALID_XY_SIZES = {1100, 1101}      # only process images with these XY dimensions
OUTPUT_BASE    = r"/home/students/rawat/Results"

# Extract model name from: .../checkpoints/<MODEL_NAME>/best.pt
_match = re.search(r'/checkpoints/([^/]+)/', CHECKPOINT_PATH)
if _match is None:
    raise ValueError(f"Cannot extract model name from CHECKPOINT_PATH:\n  {CHECKPOINT_PATH}")
MODEL_NAME = _match.group(1)

print(f"Device         : {DEVICE}")
print(f"Model name     : {MODEL_NAME}")
print(f"Checkpoint     : {CHECKPOINT_PATH}")
print(f"Input folder   : {INPUT_FOLDER}")
print(f"Output base    : {OUTPUT_BASE}/{MODEL_NAME}/")

Device         : cuda
Model name     : vit_l_lm_neo_augmented_PSF_v2
Checkpoint     : /work/scratch/rawat/models/microsam_training_output_large/checkpoints/vit_l_lm_neo_augmented_PSF_v2/best.pt
Input folder   : /home/students/rawat/Final_dataset/PSF/test/images/
Output base    : /home/students/rawat/Results/vit_l_lm_neo_augmented_PSF_v2/


In [11]:
# ============================
# 2. SCAN FOLDER FOR VALID IMAGES
# ============================
from pathlib import Path

all_tifs = sorted(Path(INPUT_FOLDER).glob("*.tif"))
valid_images = []

print(f"Scanning: {INPUT_FOLDER}\n")
for p in all_tifs:
    arr = tifffile.imread(str(p))
    y, x = arr.shape[-2], arr.shape[-1]
    if y in VALID_XY_SIZES and x in VALID_XY_SIZES:
        valid_images.append(p)
        print(f"  VALID   {p.name}  {arr.shape}")
    else:
        print(f"  SKIP    {p.name}  {arr.shape}  (XY={y}x{x})")

print(f"\n{len(valid_images)} image(s) will be processed.")

Scanning: /home/students/rawat/Final_dataset/PSF/test/images/



  SKIP    Brain1_1340_Cont_AiryscanSR_Fly1_CalyxL.tif  (70, 2, 1834, 1834)  (XY=1834x1834)
  VALID   Brain1_1340_Cont_AiryscanSR_Fly1_CalyxL_1100.tif  (70, 2, 1100, 1100)
  VALID   PN_KC_488_568_Brain2_CalyxR_LSMPlus_16bit.ome_deconv.tif  (2, 76, 1101, 1101)

2 image(s) will be processed.


In [12]:
# ============================
# 3. LOAD MODEL ONCE
# ============================
print(f"Loading model: {MODEL_NAME} ...")
predictor, segmenter = get_predictor_and_segmenter(
    model_type=MODEL_TYPE,
    checkpoint=CHECKPOINT_PATH,
    device=DEVICE,
    is_tiled=True,
)
print("Model ready.")

Loading model: vit_l_lm_neo_augmented_PSF_v2 ...
Model ready.


In [13]:
# ============================
# 4. INFERENCE LOOP — all valid images
# ============================

for img_path in valid_images:
    print(f"\n{'─'*60}")
    print(f"Processing: {img_path.name}")

    # ── Preprocess ───────────────────────────────────────────
    raw_data = tifffile.imread(str(img_path))
    if raw_data.ndim == 4:
        if raw_data.shape[0] <= 4:       # (C, Z, Y, X)
            raw_data = np.transpose(raw_data, (1, 2, 3, 0))
        elif raw_data.shape[1] <= 4:     # (Z, C, Y, X)
            raw_data = np.transpose(raw_data, (0, 2, 3, 1))
    elif raw_data.ndim == 3:
        raw_data = raw_data[..., np.newaxis]

    low, high = np.percentile(raw_data, (0.00001, 99.9999))
    if high > low:
        raw_data = np.clip((raw_data.astype(np.float32) - low) / (high - low), 0, 1) * 255
    else:
        raw_data = np.zeros_like(raw_data, dtype=np.float32)

    Z, Y, X, C = raw_data.shape
    raw_rgb = np.zeros((Z, Y, X, 3), dtype=raw_data.dtype)
    raw_rgb[..., :min(C, 2)] = raw_data[..., :min(C, 2)]
    print(f"  Preprocessed: {raw_rgb.shape}")

    # ── Slice-by-slice inference ──────────────────────────────
    prediction_stack = []
    for z in tqdm(range(Z), desc="  Slices", leave=False):
        pred_2d = automatic_instance_segmentation(
            predictor=predictor,
            segmenter=segmenter,
            input_path=raw_rgb[z],
            ndim=2,
            tile_shape=(896, 896),
            halo=(64, 64),
        )
        prediction_stack.append(pred_2d)

    predictions_3d = np.array(prediction_stack, dtype=np.uint32)

    # ── 3D connected components ───────────────────────────────
    structure_3d = generate_binary_structure(3, 3)
    stitched, n_obj = ndimage_label(predictions_3d > 0, structure=structure_3d)
    print(f"  After 3D CC: {n_obj} objects")

    # ── Blob removal ──────────────────────────────────────────
    clean = remove_small_objects(stitched, min_size=BLOB_REMOVAL_THRESHOLD, connectivity=1)
    n_after_vox = len(np.unique(clean)) - 1
    print(f"  After blob removal (>{BLOB_REMOVAL_THRESHOLD} vox): {n_after_vox} objects")

    # ── Z-stack span filter (remove objects spanning ≤3 z-slices) ─
    MIN_Z_SPAN = 3
    for lbl in np.unique(clean)[1:]:  # skip background 0
        if np.any(clean == lbl, axis=(1, 2)).sum() <= MIN_Z_SPAN:
            clean[clean == lbl] = 0
    n_final = len(np.unique(clean)) - 1
    print(f"  After z-span filter (>{MIN_Z_SPAN} z-slices): {n_final} objects")

    # ── Save ──────────────────────────────────────────────────
    out_path = os.path.join(OUTPUT_BASE, MODEL_NAME, f"{img_path.stem}_masks.tif")
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    dtype = np.uint32 if n_final > 65535 else np.uint16
    tifffile.imwrite(out_path, clean.astype(dtype))
    print(f"  Saved → {out_path}")

print(f"\n{'='*60}")
print(f"Done. Processed {len(valid_images)} image(s).")


────────────────────────────────────────────────────────────
Processing: Brain1_1340_Cont_AiryscanSR_Fly1_CalyxL_1100.tif
  Preprocessed: (70, 1100, 1100, 3)


Compute Image Embeddings 2D tiled: 100%|██████████| 4/4 [00:03<00:00,  1.20it/s]
Initialize tiled instance segmentation with decoder: 100%|██████████| 4/4 [00:00<00:00,  7.44it/s]
Compute Image Embeddings 2D tiled: 100%|██████████| 4/4 [00:03<00:00,  1.20it/s]
Initialize tiled instance segmentation with decoder: 100%|██████████| 4/4 [00:00<00:00,  7.36it/s]
Compute Image Embeddings 2D tiled: 100%|██████████| 4/4 [00:03<00:00,  1.19it/s]
Initialize tiled instance segmentation with decoder: 100%|██████████| 4/4 [00:00<00:00,  7.38it/s]
Compute Image Embeddings 2D tiled: 100%|██████████| 4/4 [00:03<00:00,  1.19it/s]
Initialize tiled instance segmentation with decoder: 100%|██████████| 4/4 [00:00<00:00,  7.34it/s]
Compute Image Embeddings 2D tiled: 100%|██████████| 4/4 [00:03<00:00,  1.18it/s]
Initialize tiled instance segmentation with decoder: 100%|██████████| 4/4 [00:00<00:00,  7.33it/s]
Compute Image Embeddings 2D tiled: 100%|██████████| 4/4 [00:03<00:00,  1.19it/s]
Initialize tiled in

  After 3D CC: 48 objects
  After blob removal (>862 vox): 23 objects
  After z-span filter (>3 z-slices): 18 objects
  Saved → /home/students/rawat/Results/vit_l_lm_neo_augmented_PSF_v2/Brain1_1340_Cont_AiryscanSR_Fly1_CalyxL_1100_masks.tif

────────────────────────────────────────────────────────────
Processing: PN_KC_488_568_Brain2_CalyxR_LSMPlus_16bit.ome_deconv.tif
  Preprocessed: (76, 1101, 1101, 3)


Compute Image Embeddings 2D tiled: 100%|██████████| 4/4 [00:03<00:00,  1.17it/s]
Initialize tiled instance segmentation with decoder: 100%|██████████| 4/4 [00:00<00:00,  7.18it/s]
Compute Image Embeddings 2D tiled: 100%|██████████| 4/4 [00:03<00:00,  1.16it/s]
Initialize tiled instance segmentation with decoder: 100%|██████████| 4/4 [00:00<00:00,  6.69it/s]
Compute Image Embeddings 2D tiled: 100%|██████████| 4/4 [00:03<00:00,  1.14it/s]
Initialize tiled instance segmentation with decoder: 100%|██████████| 4/4 [00:00<00:00,  6.85it/s]
Compute Image Embeddings 2D tiled: 100%|██████████| 4/4 [00:03<00:00,  1.14it/s]
Initialize tiled instance segmentation with decoder: 100%|██████████| 4/4 [00:00<00:00,  6.88it/s]
Compute Image Embeddings 2D tiled: 100%|██████████| 4/4 [00:03<00:00,  1.13it/s]
Initialize tiled instance segmentation with decoder: 100%|██████████| 4/4 [00:00<00:00,  6.66it/s]
Compute Image Embeddings 2D tiled: 100%|██████████| 4/4 [00:03<00:00,  1.11it/s]
Initialize tiled in

  After 3D CC: 52 objects
  After blob removal (>862 vox): 19 objects
  After z-span filter (>3 z-slices): 17 objects
  Saved → /home/students/rawat/Results/vit_l_lm_neo_augmented_PSF_v2/PN_KC_488_568_Brain2_CalyxR_LSMPlus_16bit.ome_deconv_masks.tif

Done. Processed 2 image(s).
